<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/San%20Gabriel/Versi%C3%B3n%202/SGB_Plantilla_ver3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MARCO GEOESPACIAL

## Área Natural No Protegida: Bosque de San Gabriel

**Tesista:** Naylet Hernández Sánchez  
**Director:** Viacheslav Shalisko  
**Doctorado:** Geografía y Ordenamiento Territorial  
**Proyecto:** "Evaluación comparativa de la pérdida de hábitat en territorios protegidos y no protegidos de Jalisco mediante percepción remota y aprendizaje automatizado en el periodo 2000-2020".

## Descripción del área

**1. Localización geográfica y contexto territorial**


**2. Características fisiográficas y geomorfológicas**


**3. Clima**



**4. Suelos y geología**



**5. Cobertura vegetal y uso del suelo**



**6. Biodiversidad y dinámica ecológica**



**7. Relevancia ambiental**


##Definir Variables

In [2]:
#  DEFINICIÓN DE VARIABLES

# Prefijo del área
PREFIJO     = 'SGB'

# Nombre del área
NOMBRE      = 'San_Gabriel'

#RUTA_GPKG  = '/content/drive/MyDrive/ANP/NoANP/NOANP-Colima.gpkg'
RUTA_GPKG  = '/content/drive/MyDrive/Colab Data/Naylet/gpkg/NOANP-Colima.gpkg'
LAYER_NAME  = None

# Carpeta de salida en Drive
#RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI/San Gabriel'
RUTA_AOI    = '/content/drive/MyDrive/Colab Data/Naylet/AOI'

# Rutas derivadas automáticamente (no tocar)
RUTA_GEOJSON= f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_BBOX   = f'{RUTA_AOI}/{PREFIJO}_bbox.geojson'
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'
RUTA_BBOX_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas_bbox.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
#GEE_PROJECT = 'ee-nayleths'
GEE_PROJECT = 'ee-vshalisko'

# Expansion rectangulo
## Distancia en m que se agrega a los limites de coordenadas de cada lado del
## rectangulo envolvente
buffer = 1500

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo (bounding box)
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio'

In [3]:

import ee
import os
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Mounted at /content/drive
✓ Entorno listo


In [4]:
# Crear carpeta AOI si no existe
os.makedirs(RUTA_AOI, exist_ok=True)

# Cargar y reproyectar a geográfico
gdf = gpd.read_file(RUTA_GPKG, layer=LAYER_NAME).to_crs(epsg=CRS_GEO)
minx, miny, maxx, maxy = gdf.total_bounds

# Reproyectar a UTM
gdf_utm = gdf.to_crs(epsg=CRS_UTM)
minx_u, miny_u, maxx_u, maxy_u = gdf_utm.total_bounds

# Guardar AOI como GeoJSON en Drive
gdf.to_file(RUTA_GEOJSON, driver='GeoJSON')
print(f"AOI guardado en: {RUTA_GEOJSON}")

# Guardar CSV
df = pd.DataFrame([{
    "Area"     : NOMBRE,
    "Lat_min"  : miny,
    "Lat_max"  : maxy,
    "Lon_min"  : minx,
    "Lon_max"  : maxx,
    "UTM_X_min": minx_u,
    "UTM_X_max": maxx_u,
    "UTM_Y_min": miny_u,
    "UTM_Y_max": maxy_u,
    "AOI"      : RUTA_GEOJSON
}])
df.to_csv(RUTA_CSV, index=False)
print(f"CSV guardado en: {RUTA_CSV}")

# Mostrar resultados

html = f"""
<div style="font-family: sans-serif; max-width: 420px;">

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px; margin-bottom: 12px;">
    <b style="font-size: 15px;">Sistema Geodésico: WGS84</b><br><br>
    Latitud: {miny:.6f} – {maxy:.6f}<br>
    Longitud: {minx:.6f} – {maxx:.6f}
  </div>

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px;">
    <b style="font-size: 15px;">Sistema Proyectado: UTM Zona 13N</b><br><br>
    X: {minx_u:.2f} – {maxx_u:.2f}<br>
    Y: {miny_u:.2f} – {maxy_u:.2f}
  </div>

</div>
"""
display(HTML(html))

AOI guardado en: /content/drive/MyDrive/Colab Data/Naylet/AOI/aoi_SGB.geojson
CSV guardado en: /content/drive/MyDrive/Colab Data/Naylet/AOI/SGB_coordenadas.csv


In [5]:
# Cargar el área de estudio y reproyectar a WGS84
#gdf = gpd.read_file(RUTA_GEOJSON, layer=LAYER_NAME).to_crs(epsg=CRS_GEO)

# Calcular expanded bounding box en UTM
minx_b_u = minx_u - buffer
miny_b_u = miny_u - buffer
maxx_b_u = maxx_u + buffer
maxy_b_u = maxy_u + buffer

bbox_gdf_utm = gpd.GeoDataFrame(
    geometry=[box(minx_b_u, miny_b_u, maxx_b_u, maxy_b_u)], crs=CRS_UTM
)

bbox_gdf = bbox_gdf_utm.to_crs(epsg=CRS_GEO)
minx_b_g, miny_b_g, maxx_b_g, maxy_b_g = bbox_gdf.total_bounds

# Guardar rectangulo delimitador como GeoJSON en Drive
bbox_gdf.to_file(RUTA_BBOX, driver='GeoJSON')

# Guardar CSV
df_bbox = pd.DataFrame([{
    "Area"     : NOMBRE,
    "Lat_min"  : miny_b_g,
    "Lat_max"  : maxy_b_g,
    "Lon_min"  : minx_b_g,
    "Lon_max"  : maxx_b_g,
    "UTM_X_min": minx_b_u,
    "UTM_X_max": maxx_b_u,
    "UTM_Y_min": miny_b_u,
    "UTM_Y_max": maxy_b_u,
    "AOI"      : RUTA_BBOX
}])
df_bbox.to_csv(RUTA_BBOX_CSV, index=False)
print(f"CSV guardado en: {RUTA_BBOX_CSV}")

# Mostrar resultados

html = f"""
<div style="font-family: sans-serif; max-width: 420px;">

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px; margin-bottom: 12px;">
    <b style="font-size: 15px;">BBOX Sistema Geodésico: WGS84</b><br><br>
    Latitud: {miny_b_g:.6f} – {maxy_b_g:.6f}<br>
    Longitud: {minx_b_g:.6f} – {maxx_b_g:.6f}
  </div>

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px;">
    <b style="font-size: 15px;">BBOX Sistema Proyectado: UTM Zona 13N</b><br><br>
     X: {minx_b_u:.2f} – {maxx_b_u:.2f}<br>
    Y: {miny_b_u:.2f} – {maxy_b_u:.2f}
  </div>

</div>
"""
display(HTML(html))

# Calcular centroide en WGS84
# FIX: calcular centroide en CRS proyectado, luego volver a WGS84
centroid_proj = gdf.to_crs(epsg=3857).dissolve().centroid.iloc[0]

centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]

CSV guardado en: /content/drive/MyDrive/Colab Data/Naylet/AOI/SGB_coordenadas_bbox.csv


In [ ]:
#  Crear el mapa interactivo
mapa = geemap.Map()
mapa.set_center(centroid.x, centroid.y, ZOOM)

# Agregar área de estudio
mapa.add_gdf(gdf, layer_name=MAP_TITLE, style=STYLE_AREA)

# Agregar rectángulo rojo (bounding box)
mapa.add_gdf(bbox_gdf, layer_name='Bounding box', style=STYLE_BBOX)
print("Representación espacial")
# Mostrar el mapa
mapa